## Assignment 4: Language Models and Inference

Welcome to the fourth assignment of CS 541! In Assignment 3 the agent searched a game tree. Here the "game" is writing text: a pre-trained language model (LM) tells you, for any prefix, how likely each next token is, and everything you do with it is *inference* on top of that one distribution. You will (1) turn the LM into a sentiment classifier by reading off the probabilities of two keywords, (2) generate text with beam search, the same limited-memory search from Lecture 4, and (3) change one line of that algorithm to get nucleus sampling. This assignment must be done individually.

After this assignment, you should be able to:
1. Read a probability for a label off a pre-trained LM's next-token distribution, and explain why that probability is biased.
2. Implement beam search over tokens and verify it against a reference implementation.
3. Modify the generation loop into nucleus (top-$p$) sampling, and describe the likelihood versus diversity trade-off between greedy, beam and sampling decoders.

In CS541, any work generated by an AI shouldn't be included without declaration. If you include material generated by an AI, the level of AI use should be properly documented, and the actual tool should be noted (e.g., "I used Codex to proofread the codes and draft the analysis").

This assignment downloads a small LM, `Qwen/Qwen3-0.6B-Base` (about 1.2 GB, once), and the SST-2 sentiment dataset, so you need an internet connection the first time you run the cell below. It needs `torch`, `transformers` (version 4.56 or newer) and `datasets`; if an import fails, run `pip install -U torch transformers datasets` and restart the kernel. On a laptop CPU the whole notebook takes roughly 20 minutes (the sampling runs in Part 3 are the slowest); a GPU, for example a free Colab one, is picked up automatically and makes it much faster. We use a *base* model: it has only been trained to continue text, not to follow instructions. Run the cell below first.

In [ ]:
# Given -- do not modify.
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

MODEL_NAME = "Qwen/Qwen3-0.6B-Base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).eval().to(DEVICE)
EOS = tok.eos_token_id          # the end-of-text token <|endoftext|>


def prompt_ids(text):
    """The prompt as a plain list of token ids."""
    return tok(text).input_ids


@torch.no_grad()
def next_logprobs(ids):
    """ids: LongTensor (batch, length). Returns a CPU float tensor (batch, vocab) holding
    log P(next token | prefix) for every row. One forward pass; no cache."""
    logits = model(ids.to(DEVICE)).logits[:, -1, :]
    return logits.float().log_softmax(-1).cpu()


print("vocab size:", len(tok), "| EOS id:", EOS, "| device:", DEVICE)

### 1. The LM as a classifier (7')
An LM defines $P(w \mid \text{prefix})$ for every token $w$. To classify, write the question as a prefix and *read the answer off the distribution*: for a movie review, the prefix `Review: <sentence>\nSentiment:` should make the LM put high probability on ` positive` after a good review and on ` negative` after a bad one. No parameter is trained.

The data is a fixed sample of 200 sentences from the SST-2 validation set (label 1 is positive, 0 is negative; exactly half each), loaded in the cell below. Note the leading space in ` positive`: in the tokenizer ` positive` and `positive` are different tokens, and the space-prefixed one is what follows a colon in running text. The given `first_token_id(word)` returns the id of the first token of a word (several of our words are one token, but this keeps the code safe for the others).

1.1 Implement `keyword_score(sentence, pos_words, neg_words, template)`. Fill `template` with the sentence, get the next-token log-probabilities with `next_logprobs`, and return a pair `(log_odds, mass)`. Let $P_+$ be the **sum** of the probabilities of the first tokens of the words in `pos_words`, and $P_-$ the same for `neg_words`. Then `log_odds` is $\log P_+ - \log P_-$ (positive means "predict positive") and `mass` is $P_+ + P_-$, the total probability the LM put on your keywords.

1.2 Run the classifier on `EVAL_SET` with the keywords `[" positive"]` and `[" negative"]`. Store the per-sentence log-odds as `scores_basic` (a numpy array of length 200), the accuracy of predicting "positive" when the log-odds are above 0 as `acc_basic`, the mean `mass` as `mass_basic`, and the fraction of sentences predicted positive as `frac_pos_basic`.

1.3 Two fixes. First, *calibrate*: keep `scores_basic` but predict positive when the score is above its **median** (`np.median`); store the accuracy as `acc_calibrated`. Second, *widen the keywords*: with `[" positive", " good", " great"]` against `[" negative", " bad", " terrible"]` and threshold 0, store the accuracy as `acc_multi`.

1.4 Write a different template of your own (a string containing `{s}`) as `my_template`, run it with the original two keywords, apply the median threshold, and store the accuracy as `acc_my_template`.

1.5 Write your analysis.

`run_classifier` below does the loop over the 200 sentences for you. Each call takes about 40 seconds on a CPU.

In [ ]:
# Given -- do not modify.
sst = load_dataset("stanfordnlp/sst2", split="validation")
_pool = [(row["sentence"].strip(), row["label"]) for row in sst]
random.Random(0).shuffle(_pool)
EVAL_SET = _pool[:200]                                    # [(sentence, label)], label 1 = positive
LABELS = np.array([label for _, label in EVAL_SET])
TEMPLATE = "Review: {s}\nSentiment:"


def first_token_id(word):
    return tok.encode(word, add_special_tokens=False)[0]


def run_classifier(pos_words, neg_words, template=TEMPLATE):
    """keyword_score on every sentence of EVAL_SET. Returns (log_odds, mass) as numpy arrays."""
    out = [keyword_score(s, pos_words, neg_words, template) for s, _ in EVAL_SET]
    return np.array([a for a, _ in out]), np.array([m for _, m in out])


print(len(EVAL_SET), "sentences,", LABELS.mean(), "positive. Example:", EVAL_SET[0])

In [ ]:
# Starter
def keyword_score(sentence, pos_words, neg_words, template=TEMPLATE):
    """Returns (log P(pos group) - log P(neg group), P(pos group) + P(neg group))."""
    # TODO -- fill the template, call next_logprobs once, then sum probabilities over each word group
    return 0.0, 0.0


# TODO -- 1.2
scores_basic = None
acc_basic = None
mass_basic = None
frac_pos_basic = None

# TODO -- 1.3
acc_calibrated = None
acc_multi = None

# TODO -- 1.4
my_template = None
acc_my_template = None

Address all four:
- Report `mass_basic`. The LM could have put all its probability on the two keywords. What does the number say about what it expects after `Sentiment:`, and why is "the probability of positive" in this classifier a *relative* quantity?
- Compare `frac_pos_basic` with the true fraction of positives. Which label does the zero-threshold classifier over-predict? Give a reason based on how often each word appears in pre-training text.
- Why does thresholding at the median help (`acc_calibrated` against `acc_basic`)? What does this fix assume about the test data, and when would it fail?
- Compare `acc_multi` and `acc_my_template` with the first two numbers. Which of the three changes (keywords, threshold, template) mattered most? What does that tell you about how fragile prompt-based classification is, compared with a classifier trained on labels as in Assignments 1 and 2?

*Your answer for 1.5:*

### 2. Beam search (9')
Now generate. Decoding is search: a *state* is the text so far, an *action* appends one token, and the cost of a path is the negative log-probability of its tokens, $-\sum_t \log P(w_t \mid w_{<t})$. Greedy decoding keeps one path. Exhaustive search keeps all $|V|^n$ of them (here $|V| \approx 150{,}000$). **Beam search** from Lecture 4 keeps the best $k$ partial paths at every depth, a bounded-memory compromise.

2.1 Implement `beam_search(prompt, beam_size, max_new_tokens, length_penalty=0.0)`. It returns a list of at most `beam_size` pairs `(tokens, score)`, best first, where `tokens` is the list of generated token ids (the prompt is excluded, and a final `EOS` is included when the hypothesis ended) and `score` is the sum of the log-probabilities of those tokens.

2.2 Run beam sizes 1, 2, 4 and 8 on all four prompts (25 new tokens). Store a DataFrame `beam_df` with exactly the columns `["prompt_id", "beam_size", "logprob", "n_tokens", "text"]`, one row per run, for the best hypothesis (`prompt_id` is the index in `PROMPTS`, `logprob` its score, `n_tokens` its length including a final EOS if present, `text` the decoded text with special tokens skipped). Plot `logprob` against `beam_size`, one line per prompt.

2.3 Run `beam_search(PROMPTS[1], 4, 25, length_penalty=p)` for $p = 0$ and $p = 1$ and store the lengths of the two best hypotheses in the dict `lengths_by_penalty` with keys `0.0` and `1.0`.

2.4 Write your analysis.

The procedure to implement, one **step** at a time (every live beam has the same length, so the beams form one batch):
- Start with one beam, `([], 0.0)`: no tokens, score 0. Keep a list `finished`, initially empty.
- Run `next_logprobs` once on the batch `prompt + beam tokens`, one row per live beam. The score of a candidate (beam, token) is the beam's score plus the token's log-probability.
- Take the $2k$ best candidates over all (beam, token) pairs, best first (`torch.topk` on the flattened scores; $k$ is `beam_size`). Walk through them in that order. A candidate whose token is `EOS` goes to `finished`, but only if its position in this list (counting from 0) is below $k$; otherwise it is dropped. Any other candidate becomes a live beam for the next step. Stop walking when $k$ live beams have been collected.
- After the step, stop if `finished` holds at least $k$ hypotheses. Otherwise continue, up to `max_new_tokens` steps.
- At the end, if `finished` holds fewer than $k$ hypotheses, add the best live beams (best first) until it holds $k$. Sort `finished` by `score / len(tokens) ** length_penalty`, best first, and return the first $k$.

Data structures to use:
- A **beam** is a pair `(tokens, score)`: a Python list of generated token ids and a float, the sum of their log-probabilities. `beams` is a list of such pairs, and `finished` is another list of the same kind of pairs.
- `next_logprobs` takes a `LongTensor` of shape `(n_beams, length)`, so build it with `torch.tensor([base + tokens for tokens, _ in beams])`, where `base = prompt_ids(prompt)`. It returns a tensor of shape `(n_beams, vocab)`.
- Add each beam's score to its row of log-probabilities by making the scores a column vector: `torch.tensor([score for _, score in beams]).unsqueeze(1)` broadcasts over the vocabulary.
- `total.flatten().topk(2 * k)` returns `.values` and `.indices`. A flat index `i` came from beam `i // vocab` and token `i % vocab`; `divmod(i, vocab)` gives both. Loop with `enumerate` so that you know each candidate's rank.
- Build the next `beams` as a new list. Do not modify a beam's token list in place, because several candidates can extend the same beam: use `tokens + [t]`.
- You do not need a key-value cache; recomputing the prefix at every step is fine at this length.

In [ ]:
# Given -- do not modify.
PROMPTS = [
    "The best way to learn a new language is",
    "Q: What is the capital of France?\nA:",
    "def fibonacci(n):\n",
    "The meeting is adjourned.",
]


def strip_eos(ids):
    return [t for t in ids if t != EOS]

In [ ]:
# Starter
def beam_search(prompt, beam_size, max_new_tokens, length_penalty=0.0):
    """Returns up to beam_size (generated_token_ids, score) pairs, best first."""
    base = prompt_ids(prompt)           # the prompt as a list of token ids
    beams = [([], 0.0)]                 # live beams: (generated token ids, sum of log-probs)
    finished = []                       # finished hypotheses, the same (tokens, score) pairs
    for _ in range(max_new_tokens):
        # TODO -- one batched next_logprobs call over all live beams, add the beam scores,
        #         take the 2k best (beam, token) candidates from the flattened scores,
        #         then walk them best first with enumerate (the rank decides which EOS candidates count)
        pass
        # TODO -- stop early once `finished` holds beam_size hypotheses
    # TODO -- fill `finished` up from the live beams if needed, sort by score / len(tokens) ** length_penalty
    return []


# TODO -- 2.2, then the plot
beam_df = None

# TODO -- 2.3
lengths_by_penalty = None

Address all four (print `beam_df["text"]` or look at the decoded best hypotheses to answer):
- Beam size 1 is greedy decoding. Does the log-probability of the best hypothesis always rise with the beam size in your table? Is that guaranteed, and does any beam size guarantee the most probable sequence?
- Look at the texts for `PROMPTS[3]` and `PROMPTS[1]` at beam size 1 and at beam size 4 or more. What does greedy do, what does beam search do instead, and is the higher-probability text the better text?
- Compare `lengths_by_penalty[0.0]` and `lengths_by_penalty[1.0]`. With `length_penalty=0` the score is a plain sum of negative numbers. Explain why that favours short hypotheses and what the penalty changes.
- Each step costs one forward pass over $k$ beams. Compare the work of beam size $k$ with greedy decoding, and with enumerating every sequence of 25 tokens. What does beam search give up in exchange?

*Your answer for 2.4:*

### 3. From beam search to nucleus sampling (9')
Beam search is deterministic: ask twice, get the same text, and that text is the likeliest one the search can find. Real text is not the likeliest text; people say surprising things. **Sampling** draws the next token from the model's distribution instead of maximizing it. Sampling from the full distribution has the opposite problem: a vocabulary of 150,000 tokens has a long tail of nonsense tokens whose total mass is not small. **Nucleus sampling** (top-$p$) repairs it: keep the smallest set of most likely tokens whose probabilities add up to at least $p$, renormalize inside that set, and sample from it. The set grows when the model is unsure and shrinks to a single token when it is sure.

You only change the part of the beam search loop that picks what comes next: instead of keeping the $k$ best candidates, draw one token per sequence from the filtered distribution.

3.1 Implement `top_p_filter(probs, p)`. `probs` is a tensor of shape `(batch, vocab)` whose rows are distributions. Return a tensor of the same shape, in the **original vocabulary order**, in which every token outside the nucleus has probability 0 and the nucleus is renormalized to sum to 1. The nucleus of a row is the smallest set of top tokens whose total probability is at least `p`. Equivalently: sort descending, and keep a token if the cumulative probability of the tokens *before* it is below `p`. For the row `[0.5, 0.3, 0.1, 0.07, 0.03]` and `p = 0.75` the result is `[0.625, 0.375, 0, 0, 0]`; with `p = 1` the row is unchanged.

3.2 Implement `sample_generate(prompt, top_p, num_samples, max_new_tokens, temperature=1.0, seed=0)`, which generates `num_samples` independent continuations as one batch.

3.3 Check the limit. Store in `nucleus_equals_greedy` whether `sample_generate(PROMPTS[0], 1e-9, 1, 30)` produces the same tokens as greedy decoding (`beam_search(PROMPTS[0], 1, 30)`), after `strip_eos` on both.

3.4 Build `nucleus_df`, a DataFrame with the rows returned by `summarize(p)` for $p \in \{0.1, 0.5, 0.9, 0.95, 1.0\}$ (the given function runs 20 samples of 30 tokens). Plot `distinct_2` against `mean_logprob`, one point per $p$, labelled with $p$.

3.5 Store the per-step nucleus sizes for $p = 0.9$ as `sizes_p90` (the third return value of `sample_generate(PROMPTS[0], 0.9, 20, 30, seed=0)`) and plot them against the step with a log-scaled y axis. Store `temperature_row = summarize(1.0, temperature=0.7)`.

3.6 Write your analysis.

`sample_generate` specification:
- Create `g = torch.Generator().manual_seed(seed)` once, before the loop. This makes the result reproducible, and the grader depends on it: draw with exactly one call `torch.multinomial(filtered, 1, generator=g)` per step, on the `(num_samples, vocab)` filtered matrix.
- Each step, run `next_logprobs` on the `num_samples` sequences so far (they all have the same length). Let `lp` be that result. The sampling distribution is `softmax(lp / temperature)`, filtered with `top_p_filter`. Temperature is applied before the filter.
- Once a sample has produced `EOS` it keeps producing `EOS` for the remaining steps.
- Return `(tokens, logprobs, sizes)`: `tokens` is a `LongTensor` of shape `(num_samples, max_new_tokens)`; `logprobs` is a float tensor of the same shape holding, for each sampled token, its log-probability under the **unfiltered, temperature-1** distribution `lp`; `sizes` is a list of `max_new_tokens` floats, where entry $t$ is the mean over samples of the number of tokens with nonzero probability after filtering at step $t$.

In [ ]:
# Given -- do not modify.
def summarize(top_p, temperature=1.0, prompt=PROMPTS[0], num_samples=20, max_new_tokens=30, seed=0):
    """One row of statistics for a sampling setting. mean_logprob is the model's own average
    log-probability of the sampled tokens (higher: safer text). distinct_2 is the fraction of
    token bigrams, over all samples, that are unique (higher: more varied text)."""
    tokens, logprobs, sizes = sample_generate(prompt, top_p, num_samples, max_new_tokens, temperature, seed)
    seqs = [tuple(r) for r in tokens.tolist()]
    bigrams = [(r[i], r[i + 1]) for r in seqs for i in range(len(r) - 1)]
    return {"top_p": top_p, "mean_logprob": logprobs.mean().item(),
            "distinct_2": len(set(bigrams)) / len(bigrams),
            "unique_samples": len(set(seqs)), "mean_nucleus_size": float(np.mean(sizes))}

In [ ]:
# Starter
def top_p_filter(probs, p):
    """Zero out everything outside the top-p nucleus of each row, then renormalize."""
    # TODO
    return None


def sample_generate(prompt, top_p, num_samples, max_new_tokens, temperature=1.0, seed=0):
    """Returns (tokens, logprobs, sizes) as specified in the text above."""
    # TODO -- the beam search loop with the "keep the k best" step replaced by sampling
    return None, None, None


# TODO -- 3.3
nucleus_equals_greedy = None

# TODO -- 3.4, then the plot
nucleus_df = None

# TODO -- 3.5, then the plot
sizes_p90 = None
temperature_row = None

Address all four (sample a few texts with `tok.decode(sample_generate(...)[0][i])` to see what the numbers mean):
- Describe how `mean_logprob`, `distinct_2` and `unique_samples` move as $p$ grows from 0.1 to 1.0. Where do greedy and beam search from Part 2 sit on this trade-off, and why is `unique_samples` close to 1 for small $p$?
- At $p = 1.0$ the average nucleus is tens of thousands of tokens and the text often drifts into other languages or into noise. Explain why drawing from the full distribution does this even though every single tail token is unlikely.
- Use the `sizes_p90` plot. The nucleus size changes by more than an order of magnitude from step to step. Which steps give a small nucleus and which a large one, and why? Why can a fixed top-$k$ filter (always keep the $k$ best tokens) not adapt like this?
- Compare `temperature_row` with the rows of `nucleus_df`. Temperature 0.7 without a nucleus filter changes the probabilities of all tokens; top-$p$ removes some tokens entirely. Which row is it closest to, and what can still go wrong with temperature alone?

*Your answer for 3.6:*